In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib import gridspec

# ?? Data ?????????????????????????????????????????????????????????????????????
# Frameworks × LLMs (4 LLMs each), then BioChirp (single)
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

# shape: (framework, llm, question)  ? 4 × 4 × 7
data_raw = np.array([
    # PydanticAI
    [[52,0,0,12,13,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855]],
    # LangChain
    [[52,0,0,12,12,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855]],
    # CrewAI
    [[52,0,0,12,12,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855]],
    # PhiData
    [[52,0,0,12,12,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855],
     [52,0,0,12, 0,0,9855]],
])

# BioChirp values per question
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ?? Color setup ??????????????????????????????????????????????????????????????
# Nature-appropriate: white ? soft teal ? deep slate
colors_seq = ['#f7fbff', '#c6dbef', '#4393c3', '#08519c', '#08306b']
cmap = LinearSegmentedColormap.from_list('nature_blue', colors_seq, N=256)

# True log-scale norm: log1p so 0 maps cleanly
from matplotlib.colors import LogNorm

lognorm = LogNorm(vmin=1, vmax=9855)

def get_color(val):
    if val == 0:
        return '#f7fbff'   # near-white for zeros
    return cmap(lognorm(val))

# ?? Layout ???????????????????????????????????????????????????????????????????
# 4 frameworks (4 cols each) + 1 BioChirp col + category label col
n_fw   = len(frameworks)       # 4
n_llm  = len(llms)             # 4
n_q    = len(questions)        # 7
n_cols = n_fw * n_llm + 1 + 1  # 4*4 + BioChirp + label = 18

fig_w = 8.5
fig_h = 3.8

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

cell_w = 0.82
cell_h = 0.90
pad    = 0.10   # gap between framework groups

def col_x(fw_idx, llm_idx):
    return fw_idx * (n_llm * cell_w + pad) + llm_idx * cell_w

biochirp_x = n_fw * (n_llm * cell_w + pad) + 0.1
cat_label_x = biochirp_x + cell_w + 0.35

total_w = cat_label_x + 1.5
total_h = n_q * cell_h

# ?? Draw cells ???????????????????????????????????????????????????????????????
for q_i, q in enumerate(questions):
    row_y = (n_q - 1 - q_i) * cell_h

    for fw_i, fw in enumerate(frameworks):
        for ll_i, ll in enumerate(llms):
            val = data_raw[fw_i, ll_i, q_i]
            color = get_color(val)
            c_val = 0 if val == 0 else lognorm(val)

            x = col_x(fw_i, ll_i)
            rect = mpatches.FancyBboxPatch(
                (x + 0.03, row_y + 0.03),
                cell_w - 0.06, cell_h - 0.06,
                boxstyle='round,pad=0.03',
                linewidth=0,
                facecolor=color,
            )
            ax.add_patch(rect)

            txt_c = 'white' if c_val > 0.55 else ('#1a3a5c' if c_val > 0.1 else '#aaaaaa')
            fs = 6.2
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val),
                    ha='center', va='center', fontsize=fs,
                    color=txt_c, fontweight='medium',
                    fontfamily='sans-serif')

    # BioChirp column
    bc_val = biochirp[q_i]
    color  = get_color(bc_val)
    c_val  = lognorm(bc_val)
    rect = mpatches.FancyBboxPatch(
        (biochirp_x + 0.03, row_y + 0.03),
        cell_w - 0.06, cell_h - 0.06,
        boxstyle='round,pad=0.03',
        linewidth=0,
        facecolor=color,
    )
    ax.add_patch(rect)
    txt_c = 'white' if c_val > 0.55 else ('#1a3a5c' if c_val > 0.1 else '#aaaaaa')
    ax.text(biochirp_x + cell_w/2, row_y + cell_h/2, str(bc_val),
            ha='center', va='center', fontsize=6.2,
            color=txt_c, fontweight='medium', fontfamily='sans-serif')

    # Q label
    ax.text(-0.55, row_y + cell_h/2, q,
            ha='right', va='center', fontsize=8,
            color='#333333', fontfamily='sans-serif')

# ?? Category brackets ????????????????????????????????????????????????????????
cat_colors = {'Disease treated\nby aspirin': '#5d8aa8',
              'Drugs for\nCML':              '#5a7a6e',
              'Drug targeting\nEGFR':        '#7a6e8a'}

cat_items = list(categories.items())
for cat_name, qs in cat_items:
    idxs   = [questions.index(q) for q in qs]
    top_q  = min(idxs)
    bot_q  = max(idxs)
    y_top  = (n_q - top_q) * cell_h
    y_bot  = (n_q - bot_q - 1) * cell_h
    mid_y  = (y_top + y_bot) / 2

    brace_x = cat_label_x + 0.1
    c = cat_colors[cat_name]

    # vertical bar
    ax.plot([brace_x, brace_x], [y_bot + 0.08, y_top - 0.08],
            color=c, linewidth=2.5, solid_capstyle='round')
    # tick marks
    for y_tick in [y_bot + 0.08, y_top - 0.08]:
        ax.plot([brace_x, brace_x + 0.12], [y_tick, y_tick],
                color=c, linewidth=2.0, solid_capstyle='round')

    ax.text(brace_x + 0.22, mid_y, cat_name,
            ha='left', va='center', fontsize=7.5,
            color='#222222', fontfamily='sans-serif',
            linespacing=1.35)

# ?? Column headers: LLM labels ????????????????????????????????????????????????
header_y = total_h + 0.15
llm_short = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
llm_colors = ['#b5651d', '#2a7ab5', '#2e8b57', '#8b2252']

for fw_i, fw in enumerate(frameworks):
    # Framework name
    fw_center_x = col_x(fw_i, 0) + (n_llm * cell_w) / 2 - cell_w/2
    ax.text(fw_center_x + cell_w/2, header_y + 0.55, fw,
            ha='center', va='bottom', fontsize=8,
            fontweight='bold', color='#222222', fontfamily='sans-serif')

    # Underline for framework
    x_start = col_x(fw_i, 0) + 0.05
    x_end   = col_x(fw_i, n_llm-1) + cell_w - 0.05
    ax.plot([x_start, x_end], [header_y + 0.48, header_y + 0.48],
            color='#cccccc', linewidth=0.8)

    for ll_i, (ll, lc) in enumerate(zip(llm_short, llm_colors)):
        x = col_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.12, ll,
                ha='center', va='bottom', fontsize=5.8,
                color=lc, fontfamily='sans-serif', rotation=35,
                rotation_mode='anchor')

# BioChirp header
ax.text(biochirp_x + cell_w/2, header_y + 0.55, 'BioChirp',
        ha='center', va='bottom', fontsize=8,
        fontweight='bold', color='#222222', fontfamily='sans-serif')
ax.plot([biochirp_x + 0.05, biochirp_x + cell_w - 0.05],
        [header_y + 0.48, header_y + 0.48], color='#cccccc', linewidth=0.8)

# ?? Colorbar ?????????????????????????????????????????????????????????????????
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
sm.set_array([])
cbar_ax = fig.add_axes([0.02, 0.08, 0.011, 0.28])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_ticks([1, 12, 52, 9855])
cbar.set_ticklabels(['0', '12', '52', '9855'])
cbar.ax.tick_params(labelsize=6, colors='#444444')
cbar.outline.set_visible(False)
cbar.ax.yaxis.set_tick_params(width=0.5)
cbar_ax.set_title('Count', fontsize=6, color='#444444', pad=4)

# ?? Axes cleanup ?????????????????????????????????????????????????????????????
ax.set_xlim(-0.7, total_w)
ax.set_ylim(-0.15, total_h + 1.3)
ax.axis('off')

plt.tight_layout(pad=0.3)

plt.show()
plt.savefig('nature_heatmap.pdf', dpi=300,
            bbox_inches='tight', facecolor='white')
# print("Saved.")

/tmp/ipykernel_1092025/3154564510.py:218: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0.3)
/tmp/ipykernel_1092025/3154564510.py:220: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# ─── 1. GLOBAL STYLING (90x90 mm, 5pt Arial, Vector PDF) ──────────────
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['axes.titlesize'] = 5
mpl.rcParams['axes.labelsize'] = 5
mpl.rcParams['xtick.labelsize'] = 5
mpl.rcParams['ytick.labelsize'] = 5

fig_w_in = 100 / 25.4
fig_h_in = 60 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# ─── 2. DATA ───────────────────────────────────────────────────────────
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
])
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ─── 3. COLORMAP & SCALING ─────────────────────────────────────────────
colors_seq = ['#f4f8fc', '#c6dbef', '#4393c3', '#08519c', '#08306b']
cmap = LinearSegmentedColormap.from_list('nature_blue', colors_seq, N=256)
lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    return '#f4f8fc' if val == 0 else cmap(lognorm(val))

# ─── 4. LAYOUT COORDINATES ─────────────────────────────────────────────
n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.0
cell_h = 1.4      # Slightly taller than wide makes heatmaps readable
pad_fw = 0.25     # Gap between framework groups
pad_bc = 0.65     # Gap before BioChirp

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.4

# ─── 5. DRAWING CELLS & TEXT ───────────────────────────────────────────
for q_i, q in enumerate(questions):
    row_y = get_y(q_i)
    
    # Q Labels (Left)
    ax.text(-0.4, row_y + cell_h/2, q, ha='right', va='center', fontweight='bold', color='#222222')
    
    # Framework Cells
    for fw_i in range(n_fw):
        for ll_i in range(n_llm):
            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)
            
            rect = mpatches.Rectangle((x, row_y), cell_w, cell_h, facecolor=get_color(val), edgecolor='white', lw=0.5)
            ax.add_patch(rect)
            
            txt_c = 'white' if val > 50 else ('#111111' if val > 0 else '#a0a0a0')
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val), ha='center', va='center', color=txt_c)

    # BioChirp Cell
    bc_val = biochirp[q_i]
    rect = mpatches.Rectangle((bc_x, row_y), cell_w, cell_h, facecolor=get_color(bc_val), edgecolor='white', lw=0.5)
    ax.add_patch(rect)
    
    txt_c = 'white' if bc_val > 50 else ('#111111' if bc_val > 0 else '#a0a0a0')
    ax.text(bc_x + cell_w/2, row_y + cell_h/2, str(bc_val), ha='center', va='center', color=txt_c)

# ─── 6. HEADERS (LLMs & Frameworks) ────────────────────────────────────
header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):
    start_x = get_x(fw_i, 0)
    end_x = get_x(fw_i, n_llm - 1) + cell_w
    
    # Span line and framework label
    ax.plot([start_x + 0.1, end_x - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
    ax.text((start_x + end_x) / 2, header_y + 2.0, fw, ha='center', va='bottom', fontweight='bold')
    
    # 45-degree LLM labels
    for ll_i, ll in enumerate(llms):
        x = get_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.2, ll, ha='left', va='bottom', rotation=45, rotation_mode='anchor', color='#333333')

# BioChirp Header
ax.plot([bc_x + 0.1, bc_x + cell_w - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
ax.text(bc_x + cell_w/2, header_y + 2.0, 'BioChirp', ha='center', va='bottom', fontweight='bold')

# ─── 7. CATEGORY BRACKETS (Right) ──────────────────────────────────────
for cat_name, qs in categories.items():
    idxs = [questions.index(q) for q in qs]
    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2
    
    # Minimalist bracket
    ax.plot([cat_x, cat_x], [y_bot, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot], color='#555555', lw=0.6)
    
    ax.text(cat_x + 0.35, mid_y, cat_name, ha='left', va='center', linespacing=1.3, color='#333333')

# ─── 8. CLEAN COLORBAR (Bottom Left) ───────────────────────────────────
# Placed proportionally in the negative Y space
cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['0', '10', '100', '1k', '10k'])
cbar.outline.set_linewidth(0.4)
cbar.ax.tick_params(width=0.4, length=2, pad=1, colors='#333333')
cbar_ax.set_title('Entity Count', fontsize=5, pad=3, color='#222222')

# ─── 9. PERFECT CENTERING & EXPORT ─────────────────────────────────────
# Lock the aspect ratio to prevent stretching and perfectly pad out the 90x90 canvas.
ax.set_aspect('equal')
ax.set_xlim(-1.5, cat_x + 4.5)  
ax.set_ylim(-3.5, header_y + 3.5)
ax.axis('off')
plt.tight_layout()

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.savefig('nature_heatmap_90x90.pdf', format='pdf', dpi=300, facecolor='white')
print("Saved clean 90x90 vector PDF.")

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Saved clean 90x90 vector PDF.


In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# ─── 1. GLOBAL STYLING (90x90 mm, 5pt Arial, Vector PDF) ──────────────
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['axes.titlesize'] = 5
mpl.rcParams['axes.labelsize'] = 5
mpl.rcParams['xtick.labelsize'] = 5
mpl.rcParams['ytick.labelsize'] = 5

fig_w_in = 110 / 25.4
fig_h_in = 60 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# ─── 2. DATA ───────────────────────────────────────────────────────────
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
])
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ─── 3. COLORMAP & SCALING ─────────────────────────────────────────────
colors_seq = ['#f4f8fc', '#c6dbef', '#4393c3', '#08519c', '#08306b']
cmap = LinearSegmentedColormap.from_list('nature_blue', colors_seq, N=256)
lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    return '#f4f8fc' if val == 0 else cmap(lognorm(val))

# ─── 4. LAYOUT COORDINATES (Adjusted for wider cells) ──────────────────
n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.2      # Widened cells to better fit 4 digits
cell_h = 1.4      
pad_fw = 0.15     # Reduced padding to compensate for wider cells
pad_bc = 0.4      

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.3

# ─── 5. DRAWING CELLS & TEXT ───────────────────────────────────────────
for q_i, q in enumerate(questions):
    row_y = get_y(q_i)
    
    # Q Labels (Left)
    ax.text(-0.4, row_y + cell_h/2, q, ha='right', va='center', fontweight='bold', color='#222222')
    
    # Framework Cells
    for fw_i in range(n_fw):
        for ll_i in range(n_llm):
            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)
            
            rect = mpatches.Rectangle((x, row_y), cell_w, cell_h, facecolor=get_color(val), edgecolor='white', lw=0.5)
            ax.add_patch(rect)
            
            txt_c = 'white' if val > 50 else ('#111111' if val > 0 else '#a0a0a0')
            # Dynamic font shrinking just for 9855
            dynamic_fs = 3.8 if val > 999 else 5
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val), ha='center', va='center', color=txt_c, fontsize=dynamic_fs)

    # BioChirp Cell
    bc_val = biochirp[q_i]
    rect = mpatches.Rectangle((bc_x, row_y), cell_w, cell_h, facecolor=get_color(bc_val), edgecolor='white', lw=0.5)
    ax.add_patch(rect)
    
    txt_c = 'white' if bc_val > 50 else ('#111111' if bc_val > 0 else '#a0a0a0')
    dynamic_fs = 3.8 if bc_val > 999 else 5
    ax.text(bc_x + cell_w/2, row_y + cell_h/2, str(bc_val), ha='center', va='center', color=txt_c, fontsize=dynamic_fs)

# ─── 6. HEADERS (LLMs & Frameworks) ────────────────────────────────────
header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):
    start_x = get_x(fw_i, 0)
    end_x = get_x(fw_i, n_llm - 1) + cell_w
    
    # Span line and framework label
    ax.plot([start_x + 0.1, end_x - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
    ax.text((start_x + end_x) / 2, header_y + 2.0, fw, ha='center', va='bottom', fontweight='bold')
    
    # 45-degree LLM labels
    for ll_i, ll in enumerate(llms):
        x = get_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.2, ll, ha='left', va='bottom', rotation=45, rotation_mode='anchor', color='#333333')

# BioChirp Header
ax.plot([bc_x + 0.1, bc_x + cell_w - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
ax.text(bc_x + cell_w/2, header_y + 2.0, 'BioChirp', ha='center', va='bottom', fontweight='bold')

# ─── 7. CATEGORY BRACKETS (Right) ──────────────────────────────────────
for cat_name, qs in categories.items():
    idxs = [questions.index(q) for q in qs]
    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2
    
    # Minimalist bracket
    ax.plot([cat_x, cat_x], [y_bot, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot], color='#555555', lw=0.6)
    
    ax.text(cat_x + 0.35, mid_y, cat_name, ha='left', va='center', linespacing=1.3, color='#333333')

# ─── 8. CLEAN COLORBAR (Bottom Left) ───────────────────────────────────
# Placed proportionally in the negative Y space
cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['0', '10', '100', '1k', '10k'])
cbar.outline.set_linewidth(0.4)
cbar.ax.tick_params(width=0.4, length=2, pad=1, colors='#333333')
cbar_ax.set_title('Entity Count', fontsize=5, pad=3, color='#222222')

# ─── 9. PERFECT CENTERING & EXPORT ─────────────────────────────────────
# Lock the aspect ratio to prevent stretching and perfectly pad out the 90x90 canvas.
ax.set_aspect('equal')
ax.set_xlim(-1.5, cat_x + 4.5)  
ax.set_ylim(-3.5, header_y + 3.5)
ax.axis('off')

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.savefig('nature_heatmap_fixed_9855.pdf', format='pdf', dpi=300, facecolor='white')
print("Saved clean 90x90 vector PDF with fitting numbers.")

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Saved clean 90x90 vector PDF with fitting numbers.


In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# ─── 1. GLOBAL STYLING (110x60 mm, 5pt Arial, Vector PDF) ──────────────
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['axes.titlesize'] = 5
mpl.rcParams['axes.labelsize'] = 5
mpl.rcParams['xtick.labelsize'] = 5
mpl.rcParams['ytick.labelsize'] = 5

fig_w_in = 110 / 25.4
fig_h_in = 60 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# ─── 2. DATA ───────────────────────────────────────────────────────────
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
])
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ─── 3. COLORMAP & SCALING ─────────────────────────────────────────────
# Custom gradient: Lighter mint-green transitioning up to #5BBCBF
colors_seq = ['#ebf7f6', '#bce2e1', '#8ad0d0', '#5BBCBF']
cmap = LinearSegmentedColormap.from_list('light_green_teal', colors_seq, N=256)

lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    # Pure light tint for 0 to clearly distinguish from 12
    return '#f8fcfc' if val == 0 else cmap(lognorm(val))

# ─── 4. LAYOUT COORDINATES ─────────────────────────────────────────────
n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.2      
cell_h = 1.4      
pad_fw = 0.15     
pad_bc = 0.4      

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.3

# ─── 5. DRAWING CELLS & TEXT ───────────────────────────────────────────
for q_i, q in enumerate(questions):
    row_y = get_y(q_i)
    
    # Q Labels (Left)
    ax.text(-0.4, row_y + cell_h/2, q, ha='right', va='center', fontweight='bold', color='#222222')
    
    # Framework Cells
    for fw_i in range(n_fw):
        for ll_i in range(n_llm):
            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)
            
            rect = mpatches.Rectangle((x, row_y), cell_w, cell_h, facecolor=get_color(val), edgecolor='white', lw=0.5)
            ax.add_patch(rect)
            
            # Since max color is now bright #5BBCBF, dark text reads best everywhere
            txt_c = '#111111' if val > 0 else '#a0a0a0'
            dynamic_fs = 3.8 if val > 999 else 5
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val), ha='center', va='center', color=txt_c, fontsize=dynamic_fs)

    # BioChirp Cell
    bc_val = biochirp[q_i]
    rect = mpatches.Rectangle((bc_x, row_y), cell_w, cell_h, facecolor=get_color(bc_val), edgecolor='white', lw=0.5)
    ax.add_patch(rect)
    
    txt_c = '#111111' if bc_val > 0 else '#a0a0a0'
    dynamic_fs = 3.8 if bc_val > 999 else 5
    ax.text(bc_x + cell_w/2, row_y + cell_h/2, str(bc_val), ha='center', va='center', color=txt_c, fontsize=dynamic_fs)

# ─── 6. HEADERS (LLMs & Frameworks) ────────────────────────────────────
header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):
    start_x = get_x(fw_i, 0)
    end_x = get_x(fw_i, n_llm - 1) + cell_w
    
    ax.plot([start_x + 0.1, end_x - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
    ax.text((start_x + end_x) / 2, header_y + 2.0, fw, ha='center', va='bottom', fontweight='bold')
    
    for ll_i, ll in enumerate(llms):
        x = get_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.2, ll, ha='left', va='bottom', rotation=45, rotation_mode='anchor', color='#333333')

# BioChirp Header
ax.plot([bc_x + 0.1, bc_x + cell_w - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
ax.text(bc_x + cell_w/2, header_y + 2.0, 'BioChirp', ha='center', va='bottom', fontweight='bold')

# ─── 7. CATEGORY BRACKETS (Right) ──────────────────────────────────────
for cat_name, qs in categories.items():
    idxs = [questions.index(q) for q in qs]
    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2
    
    ax.plot([cat_x, cat_x], [y_bot, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot], color='#555555', lw=0.6)
    
    ax.text(cat_x + 0.35, mid_y, cat_name, ha='left', va='center', linespacing=1.3, color='#333333')

# ─── 8. CLEAN COLORBAR (Bottom Left) ───────────────────────────────────
cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['0', '10', '100', '1k', '10k'])
cbar.outline.set_linewidth(0.4)
cbar.ax.tick_params(width=0.4, length=2, pad=1, colors='#333333')
cbar_ax.set_title('Entity Count', fontsize=5, pad=3, color='#222222')

# ─── 9. PERFECT CENTERING & EXPORT ─────────────────────────────────────
ax.set_aspect('equal')
ax.set_xlim(-1.5, cat_x + 4.5)  
ax.set_ylim(-3.5, header_y + 3.5)
ax.axis('off')

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.savefig('heatmap_light_green_to_teal.pdf', format='pdf', dpi=300, facecolor='white')
print("Saved clean vector PDF with light green gradient.")

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Saved clean vector PDF with light green gradient.


In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# ─── 1. GLOBAL STYLING (110x60 mm, 5pt Arial, Vector PDF) ──────────────
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['axes.titlesize'] = 5
mpl.rcParams['axes.labelsize'] = 5
mpl.rcParams['xtick.labelsize'] = 5
mpl.rcParams['ytick.labelsize'] = 5

fig_w_in = 110 / 25.4
fig_h_in = 60 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# ─── 2. DATA ───────────────────────────────────────────────────────────
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
])
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ─── 3. COLORMAP & SCALING ─────────────────────────────────────────────
# Custom gradient: Lighter mint-green transitioning up to #5BBCBF
colors_seq = ['#ebf7f6', '#bce2e1', '#8ad0d0', '#5BBCBF']
cmap = LinearSegmentedColormap.from_list('light_green_teal', colors_seq, N=256)

lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    return '#f8fcfc' if val == 0 else cmap(lognorm(val))

# ─── 4. LAYOUT COORDINATES (Widened for strict 5pt font) ───────────────
n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.4      # Widened significantly so "9855" fits in 5pt font
cell_h = 1.4      
pad_fw = 0.1      # Squeezed padding to make room for wider cells
pad_bc = 0.3      

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.3

# ─── 5. DRAWING CELLS & TEXT ───────────────────────────────────────────
for q_i, q in enumerate(questions):
    row_y = get_y(q_i)
    
    # Q Labels (Left)
    ax.text(-0.4, row_y + cell_h/2, q, ha='right', va='center', fontweight='bold', color='#222222')
    
    # Framework Cells
    for fw_i in range(n_fw):
        for ll_i in range(n_llm):
            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)
            
            rect = mpatches.Rectangle((x, row_y), cell_w, cell_h, facecolor=get_color(val), edgecolor='white', lw=0.5)
            ax.add_patch(rect)
            
            txt_c = '#111111' if val > 0 else '#a0a0a0'
            # Font size strictly locked to 5pt
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val), ha='center', va='center', color=txt_c, fontsize=5)

    # BioChirp Cell
    bc_val = biochirp[q_i]
    rect = mpatches.Rectangle((bc_x, row_y), cell_w, cell_h, facecolor=get_color(bc_val), edgecolor='white', lw=0.5)
    ax.add_patch(rect)
    
    txt_c = '#111111' if bc_val > 0 else '#a0a0a0'
    # Font size strictly locked to 5pt
    ax.text(bc_x + cell_w/2, row_y + cell_h/2, str(bc_val), ha='center', va='center', color=txt_c, fontsize=5)

# ─── 6. HEADERS (LLMs & Frameworks) ────────────────────────────────────
header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):
    start_x = get_x(fw_i, 0)
    end_x = get_x(fw_i, n_llm - 1) + cell_w
    
    ax.plot([start_x + 0.1, end_x - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
    ax.text((start_x + end_x) / 2, header_y + 2.0, fw, ha='center', va='bottom', fontweight='bold')
    
    for ll_i, ll in enumerate(llms):
        x = get_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.2, ll, ha='left', va='bottom', rotation=45, rotation_mode='anchor', color='#333333')

# BioChirp Header
ax.plot([bc_x + 0.1, bc_x + cell_w - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5)
ax.text(bc_x + cell_w/2, header_y + 2.0, 'BioChirp', ha='center', va='bottom', fontweight='bold')

# ─── 7. CATEGORY BRACKETS (Right) ──────────────────────────────────────
for cat_name, qs in categories.items():
    idxs = [questions.index(q) for q in qs]
    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2
    
    ax.plot([cat_x, cat_x], [y_bot, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top], color='#555555', lw=0.6)
    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot], color='#555555', lw=0.6)
    
    ax.text(cat_x + 0.35, mid_y, cat_name, ha='left', va='center', linespacing=1.3, color='#333333')

# ─── 8. CLEAN COLORBAR (Bottom Left) ───────────────────────────────────
cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['0', '10', '100', '1k', '10k'])
cbar.outline.set_linewidth(0.4)
cbar.ax.tick_params(width=0.4, length=2, pad=1, colors='#333333')
cbar_ax.set_title('Entity Count', fontsize=5, pad=3, color='#222222')

# ─── 9. PERFECT CENTERING & EXPORT ─────────────────────────────────────
ax.set_aspect('equal')
ax.set_xlim(-1.5, cat_x + 4.5)  
ax.set_ylim(-3.5, header_y + 3.5)
ax.axis('off')

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.savefig('heatmap_strict_5pt_font.pdf', format='pdf', dpi=300, facecolor='white')
print("Saved clean vector PDF with strict 5pt font.")

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Saved clean vector PDF with strict 5pt font.


In [20]:
import matplotlib
# SWITCH TO NATIVE PDF BACKEND FOR TRUE VECTORS
matplotlib.use('pdf') 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# ─── 1. GLOBAL STYLING (110x60 mm, 5pt Arial, Vector PDF) ──────────────
# Ensure text is exported as actual, editable text objects, not paths
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5
mpl.rcParams['axes.titlesize'] = 5
mpl.rcParams['axes.labelsize'] = 5
mpl.rcParams['xtick.labelsize'] = 5
mpl.rcParams['ytick.labelsize'] = 5

fig_w_in = 110 / 25.4
fig_h_in = 50 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)

# Removed the forced figure/axes facecolors to prevent giant opaque background boxes in Illustrator
ax.axis('off')

# ─── 2. DATA ───────────────────────────────────────────────────────────
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
])
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ─── 3. COLORMAP & SCALING ─────────────────────────────────────────────
colors_seq = ['#ebf7f6', '#bce2e1', '#8ad0d0', '#5BBCBF']
cmap = LinearSegmentedColormap.from_list('light_green_teal', colors_seq, N=256)
lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    return '#f8fcfc' if val == 0 else cmap(lognorm(val))

# ─── 4. LAYOUT COORDINATES ─────────────────────────────────────────────
n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.4      
cell_h = 1.4      
pad_fw = 0.1      
pad_bc = 0.3      

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.3

# ─── 5. DRAWING CELLS & TEXT (Clean Vector Patches) ────────────────────
for q_i, q in enumerate(questions):
    row_y = get_y(q_i)
    
    ax.text(-0.4, row_y + cell_h/2, q, ha='right', va='center', fontweight='bold', color='#222222')
    
    for fw_i in range(n_fw):
        for ll_i in range(n_llm):
            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)
            
            # Use completely clean vector rectangles (joinstyle ensures sharp corners)
            rect = mpatches.Rectangle((x, row_y), cell_w, cell_h, facecolor=get_color(val), edgecolor='white', lw=0.5, joinstyle='miter')
            ax.add_patch(rect)
            
            txt_c = '#111111' if val > 0 else '#a0a0a0'
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val), ha='center', va='center', color=txt_c, fontsize=5)

    bc_val = biochirp[q_i]
    rect = mpatches.Rectangle((bc_x, row_y), cell_w, cell_h, facecolor=get_color(bc_val), edgecolor='white', lw=0.5, joinstyle='miter')
    ax.add_patch(rect)
    
    txt_c = '#111111' if bc_val > 0 else '#a0a0a0'
    ax.text(bc_x + cell_w/2, row_y + cell_h/2, str(bc_val), ha='center', va='center', color=txt_c, fontsize=5)

# ─── 6. HEADERS (LLMs & Frameworks) ────────────────────────────────────
header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):
    start_x = get_x(fw_i, 0)
    end_x = get_x(fw_i, n_llm - 1) + cell_w
    
    # Capstyle keeps lines perfectly squared off in vector editors
    ax.plot([start_x + 0.1, end_x - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5, solid_capstyle='butt')
    ax.text((start_x + end_x) / 2, header_y + 2.0, fw, ha='center', va='bottom', fontweight='bold')
    
    for ll_i, ll in enumerate(llms):
        x = get_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.2, ll, ha='left', va='bottom', rotation=45, rotation_mode='anchor', color='#333333')

ax.plot([bc_x + 0.1, bc_x + cell_w - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5, solid_capstyle='butt')
ax.text(bc_x + cell_w/2, header_y + 2.0, 'BioChirp', ha='center', va='bottom', fontweight='bold')

# ─── 7. CATEGORY BRACKETS (Right) ──────────────────────────────────────
for cat_name, qs in categories.items():
    idxs = [questions.index(q) for q in qs]
    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2
    
    ax.plot([cat_x, cat_x], [y_bot, y_top], color='#555555', lw=0.6, solid_capstyle='butt')
    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top], color='#555555', lw=0.6, solid_capstyle='butt')
    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot], color='#555555', lw=0.6, solid_capstyle='butt')
    
    ax.text(cat_x + 0.35, mid_y, cat_name, ha='left', va='center', linespacing=1.3, color='#333333')

# ─── 8. COLORBAR ───────────────────────────────────────────────────────
cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['0', '10', '100', '1k', '10k'])
cbar.outline.set_linewidth(0.4)
cbar.ax.tick_params(width=0.4, length=2, pad=1, colors='#333333')
cbar_ax.set_title('Entity Count', fontsize=5, pad=3, color='#222222')

# ─── 9. PERFECT CENTERING & EXPORT ─────────────────────────────────────
ax.set_aspect('equal')
ax.set_xlim(-1.5, cat_x + 4.5)  
ax.set_ylim(-3.5, header_y + 3.5)

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
# Removed `facecolor='white'` to prevent unwanted background objects in Illustrator
plt.savefig('heatmap_pure_vector.pdf', format='pdf', dpi=300, transparent=True)
print("Saved clean, fully editable vector PDF.")

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Saved clean, fully editable vector PDF.


In [21]:
import matplotlib
matplotlib.use('svg') # SVG is much cleaner for Illustrator than PDF
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# ─── 1. GLOBAL STYLING ─────────────────────────────────────────────────
# 'none' ensures text is exported as actual editable text in SVG
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5

fig_w_in = 110 / 25.4
fig_h_in = 60 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
ax.axis('off')

# ─── 2. DATA ───────────────────────────────────────────────────────────
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855], [52,0,0,12, 0,0,9855]],
])
biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# ─── 3. COLORMAP & SCALING ─────────────────────────────────────────────
colors_seq = ['#ebf7f6', '#bce2e1', '#8ad0d0', '#5BBCBF']
cmap = LinearSegmentedColormap.from_list('light_green_teal', colors_seq, N=256)
lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    return '#f8fcfc' if val == 0 else cmap(lognorm(val))

# ─── 4. LAYOUT COORDINATES ─────────────────────────────────────────────
n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.4      
cell_h = 1.4      
pad_fw = 0.1      
pad_bc = 0.3      

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.3

# ─── 5. DRAWING CELLS & TEXT (With clip_on=False) ──────────────────────
for q_i, q in enumerate(questions):
    row_y = get_y(q_i)
    
    ax.text(-0.4, row_y + cell_h/2, q, ha='right', va='center', fontweight='bold', color='#222222', clip_on=False)
    
    for fw_i in range(n_fw):
        for ll_i in range(n_llm):
            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)
            
            # clip_on=False stops Matplotlib from wrapping this in an invisible mask
            rect = mpatches.Rectangle((x, row_y), cell_w, cell_h, facecolor=get_color(val), edgecolor='white', lw=0.5, joinstyle='miter', clip_on=False)
            ax.add_patch(rect)
            
            txt_c = '#111111' if val > 0 else '#a0a0a0'
            ax.text(x + cell_w/2, row_y + cell_h/2, str(val), ha='center', va='center', color=txt_c, fontsize=5, clip_on=False)

    bc_val = biochirp[q_i]
    rect = mpatches.Rectangle((bc_x, row_y), cell_w, cell_h, facecolor=get_color(bc_val), edgecolor='white', lw=0.5, joinstyle='miter', clip_on=False)
    ax.add_patch(rect)
    
    txt_c = '#111111' if bc_val > 0 else '#a0a0a0'
    ax.text(bc_x + cell_w/2, row_y + cell_h/2, str(bc_val), ha='center', va='center', color=txt_c, fontsize=5, clip_on=False)

# ─── 6. HEADERS ────────────────────────────────────────────────────────
header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):
    start_x = get_x(fw_i, 0)
    end_x = get_x(fw_i, n_llm - 1) + cell_w
    
    ax.plot([start_x + 0.1, end_x - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5, solid_capstyle='butt', clip_on=False)
    ax.text((start_x + end_x) / 2, header_y + 2.0, fw, ha='center', va='bottom', fontweight='bold', clip_on=False)
    
    for ll_i, ll in enumerate(llms):
        x = get_x(fw_i, ll_i) + cell_w / 2
        ax.text(x, header_y + 0.2, ll, ha='left', va='bottom', rotation=45, rotation_mode='anchor', color='#333333', clip_on=False)

ax.plot([bc_x + 0.1, bc_x + cell_w - 0.1], [header_y + 1.8, header_y + 1.8], color='#000000', lw=0.5, solid_capstyle='butt', clip_on=False)
ax.text(bc_x + cell_w/2, header_y + 2.0, 'BioChirp', ha='center', va='bottom', fontweight='bold', clip_on=False)

# ─── 7. CATEGORY BRACKETS ──────────────────────────────────────────────
for cat_name, qs in categories.items():
    idxs = [questions.index(q) for q in qs]
    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2
    
    ax.plot([cat_x, cat_x], [y_bot, y_top], color='#555555', lw=0.6, solid_capstyle='butt', clip_on=False)
    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top], color='#555555', lw=0.6, solid_capstyle='butt', clip_on=False)
    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot], color='#555555', lw=0.6, solid_capstyle='butt', clip_on=False)
    
    ax.text(cat_x + 0.35, mid_y, cat_name, ha='left', va='center', linespacing=1.3, color='#333333', clip_on=False)

# ─── 8. COLORBAR ───────────────────────────────────────────────────────
cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['0', '10', '100', '1k', '10k'])
cbar.outline.set_linewidth(0.4)
cbar.ax.tick_params(width=0.4, length=2, pad=1, colors='#333333')
cbar_ax.set_title('Entity Count', fontsize=5, pad=3, color='#222222')

# ─── 9. EXPORT AS SVG ──────────────────────────────────────────────────
ax.set_aspect('equal')
ax.set_xlim(-1.5, cat_x + 4.5)  
ax.set_ylim(-3.5, header_y + 3.5)

plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.savefig('heatmap_illustrator_friendly.svg', format='svg', dpi=300, transparent=True)
print("Saved clean, unclipped SVG for Illustrator.")

/tmp/ipykernel_1092025/2431674761.py:18: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because n

Saved clean, unclipped SVG for Illustrator.


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import math

# ─── 1. DATA PREPARATION ────────────────────────────────────────────────
df_r = pd.read_excel("~/abhi/biochirp/evaluation/Agentic_SQL/nl2SQL_result.xlsx")
df_r['Model'] = df_r['Model'].str.replace(r'grok_|openai_|gemini_|llama_', '', regex=True)

# Framework Abbreviations Mapping
fw_map = {"BioChirp": "BC", "pydanticai": "PAI", "langchain": "LC", 
          "crewai": "CAI", "phidata": "PHD"}
df_r['FW_Short'] = df_r['Framework'].map(fw_map)

# Scientific Group Mapping
df_r['Group'] = "OTHER"
df_r.loc[df_r['Question'].str.lower().str.contains('aspirin|vazalore', na=False), 'Group'] = "Diseases treated by aspirin"
df_r.loc[df_r['Question'].str.lower().str.contains('myelogenous|myeloid', na=False), 'Group'] = "DRUGS FOR CML"
df_r.loc[df_r['Question'].str.lower().str.contains('egfr|erbb1', na=False), 'Group'] = "DRUGS TARGETTING EGFR"

df_clean = df_r[df_r['Group'] != "OTHER"].copy()

# Sorting Logic for Query IDs
group_order = ["Diseases treated by aspirin", "DRUGS FOR CML", "DRUGS TARGETTING EGFR"]
df_clean['Group'] = pd.Categorical(df_clean['Group'], categories=group_order, ordered=True)
df_clean = df_clean.sort_values(by=['Group', 'Question'])

unique_q = df_clean['Question'].unique()
q_to_id = {q: f"Q{i+1}" for i, q in enumerate(unique_q)}
df_clean['Q_ID'] = df_clean['Question'].map(q_to_id)
df_clean['Q_ID'] = pd.Categorical(df_clean['Q_ID'], categories=[f"Q{i+1}" for i in range(len(unique_q))], ordered=True)

# Sort Frameworks by Latency (Fastest to Slowest)
fw_levels = df_clean.groupby('FW_Short')['latency'].median().sort_values().index.tolist()
df_clean['FW_Short'] = pd.Categorical(df_clean['FW_Short'], categories=fw_levels, ordered=True)


# ─── 2. THE 3-COLUMN CAPTION LOGIC ──────────────────────────────────────
n_q = len(unique_q)
col_size = math.ceil(n_q / 3)
q_labels = [f"Q{i+1}: {q}" for i, q in enumerate(unique_q)]

c1 = q_labels[:col_size]
c2 = q_labels[col_size:min(2 * col_size, n_q)]
c3 = q_labels[2 * col_size:] if n_q > 2 * col_size else []

max_pad = 42 
three_col_caption = "Reference Key:\n"

for i in range(col_size):
    line = f"{c1[i][:max_pad-2]:<{max_pad}}" if i < len(c1) else " " * max_pad
    line += f"{c2[i][:max_pad-2]:<{max_pad}}" if i < len(c2) else " " * max_pad
    line += f"{c3[i][:max_pad-2]}" if i < len(c3) else ""
    three_col_caption += line.rstrip() + "\n"


# ─── 3. FIGURE SETUP & PATCHWORK EQUIVALENT ──────────────────────────────
# Ensure text exports as editable vectors
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.sans-serif'] = 'Arial'

fig_w_in, fig_h_in = 180 / 25.4, 230 / 25.4
fig = plt.figure(figsize=(fig_w_in, fig_h_in), dpi=300)

# GridSpec acts like `patchwork`: 2 rows, height ratio 2.5 to 1
gs = GridSpec(2, 1, height_ratios=[2.5, 1], hspace=0.35, top=0.88, bottom=0.15)


# ─── 4. HEATMAP (SYSTEMATIC STRUCTURAL INTEGRITY) ───────────────────────
ax1 = fig.add_subplot(gs[0])

# Pivot table creates a multi-index block matching facet_grid(space="free")
pivot_df = df_clean.pivot_table(index=['FW_Short', 'Model'], columns=['Group', 'Q_ID'], values='Rows', aggfunc='first')
pivot_df = pivot_df.reindex(fw_levels, level=0) # ensure sorted rows

# Equivalent to trans="pseudo_log" and option="mako" direction=-1
norm = mcolors.SymLogNorm(linthresh=1, vmin=0, vmax=9000, base=10)
cmap = sns.color_palette("mako_r", as_cmap=True)

sns.heatmap(pivot_df, ax=ax1, cmap=cmap, norm=norm, linewidths=0.2, linecolor='white',
            cbar_kws={'label': 'Rows', 'ticks': [0, 10, 100, 1000, 9000], 'shrink': 0.8},
            annot=True, fmt='g', annot_kws={'size': 6.5, 'fontweight': 'bold'})

# Set text colors based on values (0 = Red, else White)
for text in ax1.texts:
    val = text.get_text()
    if val == '0':
        text.set_color('#FF3030')
    else:
        text.set_color('white')

# Clean up axes & add custom facet strips
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.set_xticklabels([col[1] for col in pivot_df.columns], rotation=0, fontsize=7)
ax1.set_yticklabels([row[1] for row in pivot_df.index], rotation=0, fontsize=6.5, fontstyle='italic')
ax1.set_title("A: Systematic Structural Retrieval Integrity", loc='left', fontweight='bold', fontsize=10, pad=25)

# Add Group headers (Top facet strips)
col_counts = pivot_df.columns.get_level_values(0).value_counts(sort=False)
x_pos = 0
for group, count in col_counts.items():
    ax1.axvline(x_pos, color='black', lw=1)
    mid = x_pos + count / 2
    ax1.text(mid, -0.2, group, ha='center', va='bottom', fontsize=7.5, fontweight='bold', color='white',
             bbox=dict(facecolor='#1A1A1A', edgecolor='none', boxstyle='square,pad=0.3'))
    x_pos += count

# Add Framework headers (Right facet strips)
row_counts = pivot_df.index.get_level_values(0).value_counts(sort=False).loc[fw_levels]
y_pos = 0
for fw, count in row_counts.items():
    ax1.axhline(y_pos, color='black', lw=1)
    mid = y_pos + count / 2
    ax1.text(pivot_df.shape[1] + 0.1, mid, fw, ha='center', va='center', rotation=270, 
             fontsize=7.5, fontweight='bold', color='white',
             bbox=dict(facecolor='#1A1A1A', edgecolor='none', boxstyle='square,pad=0.3'))
    y_pos += count


# ─── 5. BOXPLOT (LATENCY) ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
colors = {"BC": "#21618C", "PAI": "#7FB3D5", "LC": "#A9CCE3", "CAI": "#D4E6F1", "PHD": "#EAF2F8"}

sns.boxplot(data=df_clean, x='FW_Short', y='latency', ax=ax2, palette=colors, 
            width=0.4, linewidth=0.8, linecolor='black', fliersize=0, boxprops=dict(alpha=0.85))

sns.stripplot(data=df_clean, x='FW_Short', y='latency', ax=ax2, color='black', alpha=0.2, size=3, jitter=0.1)

# Annotate BC (Assumes BC is in fw_levels)
bc_idx = list(fw_levels).index("BC")
ax2.text(bc_idx, 14, "Superior Accuracy\nTrade-off", color="#21618C", size=7, fontweight='bold', fontstyle='italic', ha='center', va='bottom')

# Format log axis matching ggplot limits & breaks
ax2.set_yscale('log')
ax2.set_ylim(0.1, 70)
ax2.set_yticks([0.1, 1, 10, 30])
ax2.set_yticklabels(["0.1", "1", "10", "30"])

# Styling
ax2.set_xlabel("Framework", fontsize=8)
ax2.set_ylabel("Latency (sec)", fontsize=8)
ax2.set_title("B: Computational Overhead vs. Model Performance", loc='left', fontweight='bold', fontsize=10)
sns.despine(ax=ax2)


# ─── 6. TITLES & CAPTION ────────────────────────────────────────────────
fig.suptitle("Benchmark of NL2SQL Frameworks on HCDT Database", fontsize=11, fontweight='bold', ha='left', x=0.08, y=0.97)
fig.text(0.08, 0.945, "BioChirp (BC) demonstrates consistent structural retrieval superiority across drug-target groups despite computational latency.", fontsize=8.5, color="#4D4D4D")
fig.text(0.08, 0.02, three_col_caption, family='monospace', fontsize=5.8, linespacing=1.2, ha='left', va='bottom')

# Save
plt.savefig("BioChirp_Final_A4_Publication_Python.pdf", format='pdf', dpi=300, bbox_inches='tight')
print("Successfully saved PDF.")

/tmp/ipykernel_1092025/857163601.py:76: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_df = df_clean.pivot_table(index=['FW_Short', 'Model'], columns=['Group', 'Q_ID'], values='Rows', aggfunc='first')
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial


Successfully saved PDF.


In [23]:
import matplotlib
matplotlib.use('svg')  # SVG cleaner for Illustrator

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

# 1. GLOBAL STYLING
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 5

fig_w_in = 110 / 25.4
fig_h_in = 60 / 25.4
fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in), dpi=300)
ax.axis('off')

# 2. DATA
frameworks = ['PydanticAI', 'LangChain', 'CrewAI', 'PhiData']
llms       = ['Llama3', 'Claude', 'GPT-3.5', 'GPT-4']
questions  = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7']

categories = {
    'Disease treated\nby aspirin': ['Q1', 'Q2'],
    'Drugs for\nCML':              ['Q3', 'Q4', 'Q5'],
    'Drug targeting\nEGFR':        ['Q6', 'Q7'],
}

data_raw = np.array([
    [[52,0,0,12,13,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855]],
    [[52,0,0,12,12,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855], [52,0,0,12,0,0,9855]],
])

biochirp = [48, 48, 24, 24, 24, 9855, 9855]

# 3. COLORMAP WITH #F5D8DD AS MAX (LOG SCALE)

colors_seq = [
    '#ffffff',   # near zero
    '#fdecee',
    '#f8dfe4',
    '#F5D8DD'    # highest value
]

cmap = LinearSegmentedColormap.from_list(
    'pink_log_scale',
    colors_seq,
    N=256
)

lognorm = LogNorm(vmin=1, vmax=10000)

def get_color(val):
    if val == 0:
        return '#ffffff'
    return cmap(lognorm(val))

# 4. LAYOUT

n_fw, n_llm, n_q = len(frameworks), len(llms), len(questions)

cell_w = 1.4
cell_h = 1.4
pad_fw = 0.1
pad_bc = 0.3

def get_x(fw_i, ll_i):
    return fw_i * (n_llm * cell_w + pad_fw) + ll_i * cell_w

def get_y(q_i):
    return (n_q - 1 - q_i) * cell_h

bc_x = get_x(n_fw - 1, n_llm - 1) + cell_w + pad_bc
cat_x = bc_x + cell_w + 0.3

# 5. DRAW CELLS

for q_i, q in enumerate(questions):
    row_y = get_y(q_i)

    ax.text(-0.4, row_y + cell_h/2, q,
            ha='right', va='center',
            fontweight='bold',
            color='#222222',
            clip_on=False)

    for fw_i in range(n_fw):
        for ll_i in range(n_llm):

            val = data_raw[fw_i, ll_i, q_i]
            x = get_x(fw_i, ll_i)

            rect = mpatches.Rectangle(
                (x, row_y),
                cell_w,
                cell_h,
                facecolor=get_color(val),
                edgecolor='white',
                lw=0.5,
                joinstyle='miter',
                clip_on=False
            )

            ax.add_patch(rect)

            txt_c = '#111111' if val > 0 else '#a0a0a0'

            ax.text(
                x + cell_w/2,
                row_y + cell_h/2,
                str(val),
                ha='center',
                va='center',
                color=txt_c,
                fontsize=5,
                clip_on=False
            )

    bc_val = biochirp[q_i]

    rect = mpatches.Rectangle(
        (bc_x, row_y),
        cell_w,
        cell_h,
        facecolor=get_color(bc_val),
        edgecolor='white',
        lw=0.5,
        joinstyle='miter',
        clip_on=False
    )

    ax.add_patch(rect)

    txt_c = '#111111' if bc_val > 0 else '#a0a0a0'

    ax.text(
        bc_x + cell_w/2,
        row_y + cell_h/2,
        str(bc_val),
        ha='center',
        va='center',
        color=txt_c,
        fontsize=5,
        clip_on=False
    )

# 6. HEADERS

header_y = n_q * cell_h

for fw_i, fw in enumerate(frameworks):

    start_x = get_x(fw_i, 0)
    end_x   = get_x(fw_i, n_llm - 1) + cell_w

    ax.plot(
        [start_x + 0.1, end_x - 0.1],
        [header_y + 1.8, header_y + 1.8],
        color='#000000',
        lw=0.5,
        clip_on=False
    )

    ax.text(
        (start_x + end_x)/2,
        header_y + 2.0,
        fw,
        ha='center',
        va='bottom',
        fontweight='bold',
        clip_on=False
    )

    for ll_i, ll in enumerate(llms):

        x = get_x(fw_i, ll_i) + cell_w/2

        ax.text(
            x,
            header_y + 0.2,
            ll,
            ha='left',
            va='bottom',
            rotation=45,
            rotation_mode='anchor',
            color='#333333',
            clip_on=False
        )

ax.plot(
    [bc_x + 0.1, bc_x + cell_w - 0.1],
    [header_y + 1.8, header_y + 1.8],
    color='#000000',
    lw=0.5,
    clip_on=False
)

ax.text(
    bc_x + cell_w/2,
    header_y + 2.0,
    'BioChirp',
    ha='center',
    va='bottom',
    fontweight='bold',
    clip_on=False
)

# 7. CATEGORY BRACKETS

for cat_name, qs in categories.items():

    idxs = [questions.index(q) for q in qs]

    y_top = get_y(min(idxs)) + cell_h - 0.1
    y_bot = get_y(max(idxs)) + 0.1
    mid_y = (y_top + y_bot) / 2

    ax.plot([cat_x, cat_x], [y_bot, y_top],
            color='#555555',
            lw=0.6,
            clip_on=False)

    ax.plot([cat_x, cat_x + 0.15], [y_top, y_top],
            color='#555555',
            lw=0.6,
            clip_on=False)

    ax.plot([cat_x, cat_x + 0.15], [y_bot, y_bot],
            color='#555555',
            lw=0.6,
            clip_on=False)

    ax.text(
        cat_x + 0.35,
        mid_y,
        cat_name,
        ha='left',
        va='center',
        linespacing=1.3,
        color='#333333',
        clip_on=False
    )

# 8. COLORBAR

cbar_ax = ax.inset_axes([0.0, -0.16, 0.4, 0.03])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=lognorm)

cbar = fig.colorbar(
    sm,
    cax=cbar_ax,
    orientation='horizontal'
)

cbar.set_ticks([1, 10, 100, 1000, 10000])
cbar.set_ticklabels(['1', '10', '100', '1k', '10k'])

cbar.outline.set_linewidth(0.4)

cbar.ax.tick_params(
    width=0.4,
    length=2,
    pad=1,
    colors='#333333'
)

cbar_ax.set_title(
    'Entity Count',
    fontsize=5,
    pad=3,
    color='#222222'
)

# 9. EXPORT SVG

ax.set_aspect('equal')

ax.set_xlim(-1.5, cat_x + 4.5)
ax.set_ylim(-3.5, header_y + 3.5)

plt.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.99,
    bottom=0.01
)

plt.savefig(
    'heatmap_illustrator_friendly.svg',
    format='svg',
    dpi=300,
    transparent=True
)

print("Saved clean Illustrator-friendly SVG.")

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Saved clean Illustrator-friendly SVG.
